In [1]:
import os
import pandas as pd
import networkx as nx
import community as community_louvain
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyvis.network import Network

os.chdir("/Users/cansezgin/Library/Mobile Documents/com~apple~CloudDocs/Claude_Projects/Computational Social Sciences/conflict-network-analysis")

print("Working directory:", os.getcwd())

Working directory: /Users/cansezgin/Library/Mobile Documents/com~apple~CloudDocs/Claude_Projects/Computational Social Sciences/conflict-network-analysis


In [2]:
ged = pd.read_csv("data/GEDEvent_v25_1.csv")

ged['date_start'] = pd.to_datetime(ged['date_start'])

ethiopia = ged[
    (ged['country'] == 'Ethiopia') &
    (ged['date_start'] >= '2020-11-01') &
    (ged['date_start'] <= '2022-11-30')
].copy()

print(f"Total events: {len(ethiopia)}")
print(f"\nViolence types:")
print(ethiopia['type_of_violence'].value_counts())
print(f"\nSample of side_a values:")
print(ethiopia['side_a'].value_counts().head(10))
print(f"\nSample of side_b values:")
print(ethiopia['side_b'].value_counts().head(10))

Total events: 1764

Violence types:
type_of_violence
1    1203
3     554
2       7
Name: count, dtype: int64

Sample of side_a values:
side_a
Government of Ethiopia                           1405
Government of Eritrea                             158
Government of Eritrea, Government of Ethiopia      62
TPLF                                               54
OLA                                                45
Fano                                               25
Government of Somalia                               6
Amhara                                              3
OLA - Fekade Abdisa faction                         2
Burji                                               1
Name: count, dtype: int64

Sample of side_b values:
side_b
TPLF          699
Civilians     554
OLA           495
Al-Shabaab      6
Gumuz           3
GLF             3
Oromo           2
Guji            1
Nuer            1
Name: count, dtype: int64


/var/folders/ls/jfgzf4j13hx__mtdgv2_0xbw0000gn/T/ipykernel_7429/1508399459.py:1: DtypeWarning: Columns (0: gwnoa) have mixed types. Specify dtype option on import or set low_memory=False.
  ged = pd.read_csv("data/GEDEvent_v25_1.csv")


In [3]:
print("side_b values:")
print(ethiopia['side_b'].value_counts())

side_b values:
side_b
TPLF          699
Civilians     554
OLA           495
Al-Shabaab      6
Gumuz           3
GLF             3
Oromo           2
Guji            1
Nuer            1
Name: count, dtype: int64


In [4]:
# Step 1: Remove one-sided violence against civilians
battle_events = ethiopia[ethiopia['side_b'] != 'Civilians'].copy()
print(f"Events after removing civilian targeting: {len(battle_events)}")

# Step 2: Split coalition entries into individual actors
# "Government of Eritrea, Government of Ethiopia" becomes two separate actors
def split_actors(actor_string):
    """Split comma-separated actor names into a list."""
    if pd.isna(actor_string):
        return []
    parts = [a.strip() for a in actor_string.split(',')]
    # Rejoin parts that belong to one actor name
    # UCDP uses format like "Government of Ethiopia" so we check for known patterns
    merged = []
    i = 0
    while i < len(parts):
        # Check if this part starts with "Government of" - it is a complete actor
        if parts[i].startswith('Government of'):
            merged.append(parts[i])
            i += 1
        # Check if next part starts with "Government of" - current part is complete
        elif i + 1 < len(parts) and parts[i + 1].startswith('Government of'):
            merged.append(parts[i])
            i += 1
        # Check if this is the last part - it is complete
        elif i == len(parts) - 1:
            merged.append(parts[i])
            i += 1
        # Otherwise, this part and next might be one actor name
        # But in UCDP, commas separate distinct actors, so treat as separate
        else:
            merged.append(parts[i])
            i += 1
    return merged

# Test the split function
test_cases = [
    "Government of Eritrea, Government of Ethiopia",
    "TPLF",
    "OLA",
    "Government of Ethiopia"
]
for t in test_cases:
    print(f"  '{t}' -> {split_actors(t)}")

Events after removing civilian targeting: 1210
  'Government of Eritrea, Government of Ethiopia' -> ['Government of Eritrea', 'Government of Ethiopia']
  'TPLF' -> ['TPLF']
  'OLA' -> ['OLA']
  'Government of Ethiopia' -> ['Government of Ethiopia']
